# Measurements preparations

## Preamble

In [1]:
import os
from glob import glob
from datetime import datetime
import csv
import xml.etree.ElementTree as ET
import math
import re

import pandas as pd
import numpy as np
from libnmap.parser import NmapParser
import ipaddress
import pytricia

## HRP and anycast prefixes

the HRP dataset was extracted from: https://github.com/hrp-stats/hrp-stats.github.io/tree/main/hrps

In [19]:
ts = datetime(2025, 12, 14)
_anycast_ipv4_pdf = pd.read_csv(f"../anycast_census_{ts.year}_{ts.month:02d}_{ts.day:02d}_v4.csv")
anycast_ipv4_pdf = _anycast_ipv4_pdf[_anycast_ipv4_pdf["GCD_ICMPv4"] > 1].copy()
display(anycast_ipv4_pdf.head(5))
print(len(anycast_ipv4_pdf))

,prefix,AB_ICMPv4,AB_TCPv4,AB_DNSv4,GCD_ICMPv4,GCD_TCPv4,partial,backing_prefix,ASN,locations
0,1.0.0.0/24,28,28,28,68,30,False,1.0.0.0/24,13335,"[{'city': 'Bangalore', 'country_code': 'IN', '..."
1,1.1.1.0/24,28,28,27,70,29,False,1.1.1.0/24,13335,"[{'city': 'Bangalore', 'country_code': 'IN', '..."
3,1.10.10.0/24,3,0,3,2,0,False,1.10.10.0/24,148000,"[{'city': 'Delhi', 'country_code': 'IN', 'id':..."
4,1.12.0.0/24,5,0,3,5,0,False,1.12.0.0/20,132203_45090,"[{'city': 'Frankfurt-am-Main', 'country_code':..."
5,1.12.12.0/24,3,1,0,3,1,False,1.12.0.0/20,132203_45090,"[{'city': 'Hong Kong', 'country_code': 'HK', '..."


13988


In [6]:
def get_hrp_prefixes(hrp_csv_path):
    hrp_pdf = pd.read_csv(hrp_csv_path)
    hrp_pdf["prefix"] = hrp_pdf[["network_address", "prefixlen"]].apply(lambda row: f"{row['network_address']}/{row['prefixlen']}", axis=1)
    display(hrp_pdf.head(5))
    print(len(hrp_pdf))
    display(hrp_pdf.groupby("prefixlen").size().reset_index(name="counts").sort_values(by="counts", ascending=False).head(10))
    return hrp_pdf

hrp_80_pdf = get_hrp_prefixes("../hrp/80.2025-12-02.muc.slash24-hrps.csv")

,network_address,prefixlen,bgp_prefix,asn,responsive_addresses,port,date,location,prefix
0,1.11.136.0,24,1.11.128.0/17,17839,249,80,2025-12-02,muc,1.11.136.0/24
1,1.11.188.0,24,1.11.128.0/17,17839,252,80,2025-12-02,muc,1.11.188.0/24
2,1.11.197.0,24,1.11.128.0/17,17839,253,80,2025-12-02,muc,1.11.197.0/24
3,1.11.198.0,24,1.11.128.0/17,17839,238,80,2025-12-02,muc,1.11.198.0/24
4,1.33.168.0,24,1.33.0.0/16,2514,252,80,2025-12-02,muc,1.33.168.0/24


163740


,prefixlen,counts
0,24,163740


In [10]:
def compare_anycast_hrp(anycast_ipv4_pdf, hrp_pdf):
    hrp = hrp_pdf["prefix"].tolist()
    pyt = pytricia.PyTricia(32)
    for prefix in hrp:
        pyt.insert(ipaddress.IPv4Network(prefix), True)

    anycast_ipv4_pdf["is_hrp"] = anycast_ipv4_pdf["prefix"].apply(lambda x: ipaddress.IPv4Network(x) in pyt)
    hrp_anycast = len((anycast_ipv4_pdf[anycast_ipv4_pdf["is_hrp"] == True]))
    print(f"Number of anycasted IPv4 addresses that are in HRPs: {hrp_anycast}")
    print(f"Percentage of anycasted IPv4 addresses that are in HRPs: {hrp_anycast/len(anycast_ipv4_pdf)*100:.2f}%")

compare_anycast_hrp(anycast_ipv4_pdf, hrp_80_pdf)

Number of anycasted IPv4 addresses that are in HRPs: 8978
Percentage of anycasted IPv4 addresses that are in HRPs: 64.16%


In [11]:
hrp_443_pdf = get_hrp_prefixes("../hrp/443.2025-12-02.muc.slash24-hrps.csv")
compare_anycast_hrp(anycast_ipv4_pdf, hrp_443_pdf)

,network_address,prefixlen,bgp_prefix,asn,responsive_addresses,port,date,location,prefix
0,1.11.136.0,24,1.11.128.0/17,17839,246,443,2025-12-02,muc,1.11.136.0/24
1,1.11.188.0,24,1.11.128.0/17,17839,249,443,2025-12-02,muc,1.11.188.0/24
2,1.11.197.0,24,1.11.128.0/17,17839,252,443,2025-12-02,muc,1.11.197.0/24
3,1.11.198.0,24,1.11.128.0/17,17839,233,443,2025-12-02,muc,1.11.198.0/24
4,1.33.168.0,24,1.33.0.0/16,2514,250,443,2025-12-02,muc,1.33.168.0/24


164590


,prefixlen,counts
0,24,164590


Number of anycasted IPv4 addresses that are in HRPs: 9022
Percentage of anycasted IPv4 addresses that are in HRPs: 64.47%


In [14]:
hrp80 = hrp_80_pdf["prefix"].tolist()
pyt80 = pytricia.PyTricia(32)
for prefix in hrp80:
    pyt80.insert(ipaddress.IPv4Network(prefix), True)

hrp443 = hrp_443_pdf["prefix"].tolist()
pyt443 = pytricia.PyTricia(32)
for prefix in hrp443:
    pyt443.insert(ipaddress.IPv4Network(prefix), True)

anycast_ipv4_pdf["is_hrp"] = anycast_ipv4_pdf["prefix"].apply(lambda x: ipaddress.IPv4Network(x) in pyt80 or ipaddress.IPv4Network(x) in pyt443)

hrp_anycast = len(anycast_ipv4_pdf[anycast_ipv4_pdf["is_hrp"] == True])
print(f"Number of anycasted IPv4 addresses that are in HRPs (port 80 or 443): {hrp_anycast}")
print(f"Percentage of anycasted IPv4 addresses that are in HRPs (port 80 or 443): {hrp_anycast/len(anycast_ipv4_pdf)*100:.2f}%")

hrp_anycast_pdf = anycast_ipv4_pdf[anycast_ipv4_pdf["is_hrp"] == True]

Number of anycasted IPv4 addresses that are in HRPs (port 80 or 443): 9026
Percentage of anycasted IPv4 addresses that are in HRPs (port 80 or 443): 64.50%


In [ ]:
vp = "au-syd"
zmap_files = glob(f"../results/zmap/dataset=tcp-anycast/vp={vp}/**/*.csv", recursive=True)
pattern = r"vp=([^/]+)/port=([^/]+)/year=([^/]+)/month=([^/]+)/day=([^/]+)"

# nmap top 100 ports
with open("../input/zmap/nmap_100ports", "rt", encoding="utf-8") as f:
    result100 = f.read()
    localhost_ports = [int (i) for i in result100.splitlines()]

# empty file! "../results/zmap/dataset=tcp-anycast/vp=au-syd/port=113/year=2025/month=12/day=14/zmap_113_20251214091858.csv"
for zmap_file in zmap_files:
    ip_address_column = 1
    m = re.search(pattern, zmap_file)
    scan_location, scan_port, year, month, day = m.groups()
    if int(scan_port) not in localhost_ports:
        continue
    pyasndat_file = "../hrp/ipasn_20251214.dat"
    scan_date = f"{year}{month}{day}"
    output_dir = f"hrp_results/{scan_location}"

    print(f"Processing ZMap file: {os.path.basename(zmap_file)}")

    cmd = f"""
    cut -d, -f {ip_address_column} {zmap_file} | \
      tail -n +2 | \
      sort -n -t . -k 1,1 -k 2,2 -k 3,3 -k 4,4 -S10% -T /tmp --compress-program=lz4 | \
      tee {zmap_file}.sorted | \
      ../../venv/bin/python ../hrp/search-for-hrps.py -p {pyasndat_file} --scan-port {scan_port} \
          --scan-date {scan_date} --scan-location {scan_location} -o {output_dir} -
    """

    !bash -c "$cmd"
    !mv {output_dir}/slash24-info.csv.gz {output_dir}/slash24-info_{scan_location}_{scan_port}_{scan_date}.csv.gz
    !gunzip -f {output_dir}/slash24-info_{scan_location}_{scan_port}_{scan_date}.csv.gz


Processing ZMap file: zmap_113_20251214091858.csv
Traceback (most recent call last):
  File "/Users/glcesar/workspace/anycast-service-discovery/measurements/notebooks/../hrp/search-for-hrps.py", line 124, in <module>
    main()
  File "/Users/glcesar/workspace/anycast-service-discovery/measurements/notebooks/../hrp/search-for-hrps.py", line 118, in main
    find_hrps(args.input_file, args.output_dir, args.scan_port, args.scan_date, args.scan_location, asn_db, contains_headers=args.headers_included, print_all=args.all)
  File "/Users/glcesar/workspace/anycast-service-discovery/measurements/notebooks/../hrp/search-for-hrps.py", line 42, in find_hrps
    ip_str = replies.__next__()[0]
             ^^^^^^^^^^^^^^^^^^
StopIteration
mv: rename hrp_results/au-syd/slash24-info.csv.gz to hrp_results/au-syd/slash24-info_au-syd_113_20251214.csv.gz: No such file or directory
gunzip: can't stat: hrp_results/au-syd/slash24-info_au-syd_113_20251214.csv.gz (hrp_results/au-syd/slash24-info_au-syd_113_2

## IPv4

### ZMap allowlist

In [19]:
ts = datetime(2025, 12, 9)

In [20]:
_anycast_ipv4_pdf = pd.read_csv(f"../anycast_census_{ts.year}_{ts.month:02d}_{ts.day:02d}_v4.csv")
anycast_ipv4_pdf = _anycast_ipv4_pdf[_anycast_ipv4_pdf["GCD_ICMPv4"] > 1].copy()
display(anycast_ipv4_pdf.head(5))
print(len(anycast_ipv4_pdf))
anycast_ipv4_pdf["prefix"].to_csv(f"../input/zmap/anycast_prefixes_{ts.year}_{ts.month:02d}_{ts.day:02d}_v4.csv", index=False, header=False)

,prefix,AB_ICMPv4,AB_TCPv4,AB_DNSv4,GCD_ICMPv4,GCD_TCPv4,partial,backing_prefix,ASN,locations
0,1.0.0.0/24,29,29,29,69,30,False,1.0.0.0/24,13335,"[{'city': 'Dar es Salaam', 'country_code': 'TZ..."
1,1.1.1.0/24,29,29,29,68,31,False,1.1.1.0/24,13335,"[{'city': 'Bangalore', 'country_code': 'IN', '..."
3,1.10.10.0/24,3,0,3,2,0,False,1.10.10.0/24,148000,"[{'city': 'Delhi', 'country_code': 'IN', 'id':..."
4,1.12.0.0/24,5,0,5,5,0,False,1.12.0.0/20,132203,"[{'city': 'Cologne', 'country_code': 'DE', 'id..."
5,1.12.12.0/24,3,1,0,3,1,False,1.12.0.0/20,132203,"[{'city': 'Hong Kong', 'country_code': 'HK', '..."


13982


#### ZMap ports

In [ ]:
xml_files = glob("../results/nmap/*.xml")

ports = set()
for xml_file in xml_files:
    print(f"Processing {xml_file}...")
    report = NmapParser.parse_fromfile(xml_file)
    for host in report.hosts:
        for svc in host.services:
            # tcpwrapped are hosts that deliberately closed the TCP connection
            # https://secwiki.org/w/FAQ_tcpwrapped
            if svc.service in ["tcpwrapped"]:
                continue
            if svc.state == "open":
                ports.add(svc.port)


print(f"Discovered {len(ports)} open ports")

Processing ../results/nmap/nmap_20251202093456_.xml...
Discovered 1997 open ports


In [42]:
# tab delimited file
nmap_services_pdf = pd.read_csv("../nmap-services", delimiter="\t")
nmap_services_pdf.columns = ["service_name", "portnum_protocol", "open_frequency", "optional_comments"]
nmap_services_pdf.sort_values(by=["open_frequency"], ascending=False, inplace=True)
nmap_services_pdf["portnum"] = nmap_services_pdf["portnum_protocol"].apply(lambda x: int(x.split("/")[0]))
nmap_services_pdf["protocol"] = nmap_services_pdf["portnum_protocol"].apply(lambda x: x.split("/")[1])

default_nmap_ports_pdf = nmap_services_pdf[nmap_services_pdf["protocol"] == "tcp"].head(2000).copy()

default_nmap_ports = set(default_nmap_ports_pdf["portnum"].tolist())

scanned_but_missing_default = ports.difference(default_nmap_ports)
print(len(scanned_but_missing_default), "ports from measurements are not in default nmap top 2000 ports:", scanned_but_missing_default)

default_but_missing_ports = default_nmap_ports.difference(ports)
print(len(default_but_missing_ports), "default nmap ports are missing in our scan results:", default_but_missing_ports)

316 ports from measurements are not in default nmap top 2000 ports: {1, 2070, 4161, 2124, 2134, 2142, 4192, 2148, 2150, 2187, 4243, 2197, 2203, 4262, 2224, 2250, 2253, 2261, 2262, 2265, 2269, 2270, 2271, 2280, 2291, 2292, 4342, 2300, 2302, 2304, 4355, 4356, 4357, 4358, 2312, 2313, 2325, 2326, 2330, 2335, 2340, 2371, 2372, 2375, 2376, 2391, 2418, 2425, 2435, 2436, 2438, 2439, 2449, 2456, 2463, 2472, 2550, 2551, 2580, 2583, 2691, 4767, 4770, 2723, 4771, 4778, 4793, 4800, 4819, 2804, 2806, 4859, 2812, 4860, 4876, 4877, 4881, 2847, 2850, 4903, 4912, 2882, 4931, 2888, 2889, 2898, 2901, 2902, 2908, 2930, 2957, 2958, 2973, 2984, 2987, 2988, 2991, 2997, 3002, 3014, 3023, 3057, 3062, 3063, 3080, 3089, 3118, 1101, 1116, 1118, 1125, 1128, 1134, 1135, 1136, 5233, 5234, 5235, 3190, 1143, 1144, 1150, 1153, 5250, 1156, 1157, 5252, 1159, 1162, 5259, 5261, 1167, 1168, 1173, 1176, 1179, 1180, 1182, 1184, 1188, 1190, 1191, 1194, 1195, 1196, 5291, 1200, 1204, 1207, 1208, 1209, 1210, 1211, 1215, 1221, 1223

nmap scans top 1000 ports per protocol: https://nmap.org/book/man-port-specification.html

In [52]:
def parse_nmap_ports(port_string):
    """
    CHATGPT GENERATED FUNCTION
    Parse an Nmap port range string like:
    '1,3-4,6-7,9,13,17-20'
    and return a sorted list of all port numbers.
    """
    ports = set()

    for part in port_string.split(","):
        part = part.strip()
        
        # Range: e.g., "300-305"
        if "-" in part:
            start, end = part.split("-")
            start = int(start)
            end = int(end)
            ports.update(range(start, end + 1))
        
        # Single port
        else:
            ports.add(int(part))

    return set(sorted(ports))


# nmap --top-ports 2000 localhost -v -oG -
with open("nmap_localhost_2k.txt", "rt", encoding="utf-8") as f:
    result2k = f.read()

# comparing with real scans of 9 hosts...
localhost_ports = parse_nmap_ports(result2k)

print(f"Nmap default top ports count: {len(localhost_ports)}")

print(len(localhost_ports.difference(ports)))

Nmap default top ports count: 2000
3


In [ ]:
# nmap localhost -v -oG -
with open("nmap_localhost_1k.txt", "rt", encoding="utf-8") as f:
    result1k = f.read()

localhost_ports = parse_nmap_ports(result1k)

print(f"Nmap default top ports count: {len(localhost_ports)}")

with open("../input/zmap/nmap_ports", "w") as f:
    for item in localhost_ports:
        f.write(f"{item}\n")

Nmap default top ports count: 1000


### ZGrab allowlist

In [138]:
# https://github.com/zmap/zgrab2/
service_names = [s.lower() for s in ["AMQP", "BACnet", "Banner", "DNP", 
             "Fox", "FTP", "HTTP", "IMAP", "IPP",
             "JARM", "Sieve", "Memcached",
             "Modbus", "MongoDB", "MQTT", "MSSQL",
             "MySQL", "NTP", "Oracle", "POP3",
             "PostgreSQL", "PPTP", "Redis", "Siemens",
             "SMB", "SMTP", "SOCKS5", "SSH", "Telnet"]]

Finding out which protocols we scanned that matches ZGrab2

In [139]:
# nmap-services comes from /opt/homebew/share/nmap/nmap-services file (default installation of nmap on macOS)
nmap_services_pdf = pd.read_csv("../nmap-services", delimiter="\t")
nmap_services_pdf.columns = ["service_name", "portnum_protocol", "open_frequency", "optional_comments"]
nmap_services_pdf.sort_values(by=["open_frequency"], ascending=False, inplace=True)
nmap_services_pdf["portnum"] = nmap_services_pdf["portnum_protocol"].apply(lambda x: int(x.split("/")[0]))
nmap_services_pdf["protocol"] = nmap_services_pdf["portnum_protocol"].apply(lambda x: x.split("/")[1])

def in_zgrab2(x):
    for service_name in service_names:
        if service_name in x.lower().strip():
            return True
    return False

nmap_services_pdf["in_zgrab2"] = nmap_services_pdf["service_name"].apply(lambda x: in_zgrab2(x))

In [140]:
# nmap ports comes from a dry-run of nmap on localhost
with open("../input/zmap/nmap_ports", "rt", encoding="utf-8") as f:
    result = f.read().splitlines()

nmap_ports_pdf = pd.DataFrame({"portnum": [int(line.strip()) for line in result]})
print(len(nmap_ports_pdf))
# join to know if ports are in zgrab2
joined_pdf = nmap_ports_pdf.merge(nmap_services_pdf, on="portnum", how="inner")

joined_pdf[joined_pdf["in_zgrab2"] == True]

1000


,portnum,service_name,portnum_protocol,open_frequency,optional_comments,protocol,in_zgrab2
39,20,ftp-data,20/udp,0.001878,# File Transfer [Default Data],udp,True
40,20,ftp-data,20/tcp,0.001079,# File Transfer [Default Data],tcp,True
41,20,ftp-data,20/sctp,0.000000,# File Transfer [Default Data],sctp,True
42,21,ftp,21/tcp,0.197667,# File Transfer [Control],tcp,True
43,21,ftp,21/udp,0.004844,# File Transfer [Control],udp,True
...,...,...,...,...,...,...,...
1673,8080,http-proxy,8080/tcp,0.042052,# http-alt | Common HTTP proxy/second web serv...,tcp,True
1674,8080,http-alt,8080/udp,0.000000,# HTTP Alternate (see port 80),udp,True
1689,8088,radan-http,8088/tcp,0.000608,# Radan HTTP,tcp,True
1690,8088,radan-http,8088/udp,0.000000,# Radan HTTP,udp,True


In [141]:
udp_ports = joined_pdf[joined_pdf["protocol"] != "tcp"]["portnum"].tolist()
tcp_ports = nmap_ports_pdf["portnum"].tolist()

for u in udp_ports:
    if u not in tcp_ports:
        print(u)

# print nothing means we don't overlook any port. We scan all tcp ports as should.

In [142]:
def needs_tls(s, c):
    if "tls" in s.lower() or "ssl" in s.lower():
        return True
    if c is None or pd.isna(c):
        return False
    if "ssl" in c.lower() or "tls" in c.lower():
        return True
    return False

joined_pdf["needs_tls"] = joined_pdf[["service_name", "optional_comments"]].apply(lambda x: needs_tls(x["service_name"], x["optional_comments"]), axis=1)

joined_pdf[(joined_pdf["needs_tls"] == True) & (joined_pdf["in_zgrab2"] == True)].drop_duplicates(subset=["portnum"])[["portnum", "service_name", "optional_comments"]]

,portnum,service_name,optional_comments
337,443,https,# secure http (SSL)
361,465,smtps,# submissions | igmpv3lite | urd | smtp protoc...
734,990,ftps,"# ftp protocol, control, over TLS/SSL"
737,992,telnets,# telnet protocol over TLS/SSL
739,993,imaps,# imap4 protocol over TLS/SSL | IMAP over TLS ...
741,995,pop3s,# POP3 protocol over TLS/SSL | pop3 protocol o...


In [143]:
joined_pdf[(joined_pdf["needs_tls"] == True) & (joined_pdf["in_zgrab2"] == False)].drop_duplicates(subset=["portnum"])[["portnum", "service_name", "optional_comments"]]

,portnum,service_name,optional_comments
432,6699,napster,# babel-dtls | Napster File (MP3) sharing sof...
434,563,snews,# nntps | nntp protocol over TLS/SSL (was snntp)
464,636,ldapssl,# ldaps | LDAP over SSL | ldap protocol over T...
494,15002,onep-tls,# Open Network Environment TLS
573,9001,etlservicemgr,# ETL Service Manager
715,5061,sip-tls,NaN
988,1131,caspssl,# CAC App Service Protocol Encripted
1006,3077,orbix-loc-ssl,# Orbix 2000 Locator SSL
1093,3269,globalcatLDAPssl,# msft-gc-ssl | Global Catalog LDAP over ssl |...
1459,3766,sitewatch-s,# SSL e-watch sitewatch server


In [144]:
joined_pdf[(joined_pdf["needs_tls"] == False) & (joined_pdf["in_zgrab2"] == True)].drop_duplicates(subset=["portnum"])[["portnum", "service_name", "optional_comments"]]

,portnum,service_name,optional_comments
39,20,ftp-data,# File Transfer [Default Data]
42,21,ftp,# File Transfer [Control]
45,22,ssh,# Secure Shell Login
48,23,telnet,NaN
52,25,smtp,# Simple Mail Transfer
54,26,rsftp,NaN
102,2121,ccproxy-ftp,# scientia-ssdb | CCProxy FTP Proxy | SCIENTIA...
108,80,http,# World Wide Web HTTP
141,106,pop3pw,# 3com-tsmux | Eudora compatible PW changer | ...
148,110,pop3,# PostOffice V.3 | Post Office Protocol - Vers...


In [ ]:
# ports that zgrab knows but we did not scan
joined_pdf[joined_pdf["portnum"].isin([5445, 27017, 27018, 27019, 1883, 6379, 28240, 4190, 11211, 4319, 19999, 5671, 5672, 47808])]

,portnum,service_name,portnum_protocol,open_frequency,optional_comments,protocol,in_zgrab2,needs_tls


In [13]:
def set_zgrab_tag(port):
    # set the trigger of services zgrab knows and uses TLS
    if port in [443, 2381, 16993, 6789, 7443, 5989, 8443, 2381, 16993, 6789, 7443, 5989]:
        return "https"
    if port  == 465:
        return "smtps"
    if port == 990:
        return "ftps"
    if port == 993:
        return "imaps"
    if port == 995:
        return "pop3s"

    # set the trigger of services zgrab doesn't know and uses TLS
    # 3995: https://learn.microsoft.com/en-us/iis/manage/remote-administration/remote-administration-for-iis-manager
    if port in [563, 636, 15002, 9001, 5061, 1131, 3077, 3269, 3766, 5986, 3995]:
        return "bannertls"

    # set the trigger of services zgrab knows and doesn't use TLS
    if port in [20, 21, 26, 2121, 2811, 8021]:
        return "ftp"
    if port == 22:
        return "ssh"
    if port in [23, 992]:  # 992 is telnet with TLS but zgrab2 does not support it
        return "telnet"
    if port == [25, 587]:
        return "smtp"
    if port in [80, 280, 593, 16992, 6788, 4848, 777, 808, 1183, 3128, 7627, 5800, 5801, 5802, 8000, 8008, 5988, 8080, 8088]:
        return "http"
    if port == 110:
        return "pop3"
    if port == 143:
        return "imap"
    if port == 631:
        return "ipp"
    if port in [1186, 3306, 1862]:
        return "mysql"
    if port == 5432:
        return "postgres"
    if port in [1521, 2005]:
        return "oracle"
    if port == 20000:
        return "dnp3"
    if port == 1723:
        return "pptp"

    # ports 106, 119, 3920, and others go here...
    # default to banner
    return "banner"

In [21]:
def get_hrp_prefixes(pdf, port, vp):
    hrp_files = glob(f"hrp_results/{vp}/slash24-info_*_{port}_*.csv")
    hrp_port_pdf = pd.read_csv(hrp_files[0]) if hrp_files else pd.DataFrame()
    hrp_port_pdf["prefix"] = hrp_port_pdf[["network_address", "prefixlen"]].apply(lambda row: f"{row['network_address']}/{row['prefixlen']}", axis=1)
    hrp_port = hrp_port_pdf["prefix"].tolist()
    pyt = pytricia.PyTricia(32)
    for prefix in hrp_port:
        pyt.insert(ipaddress.IPv4Network(prefix), True)

    pdf["is_hrp"] = pdf["saddr"].apply(lambda x: ipaddress.IPv4Address(x) in pyt)
    return pdf

def remove_hrp(pdf, port, vp):
    hrp_pdf = get_hrp_prefixes(pdf, port, vp)
    non_hrp_pdf = hrp_pdf[hrp_pdf["is_hrp"] == False].copy()
    non_hrp_pdf.drop(columns=["is_hrp"], inplace=True)
    return non_hrp_pdf

def include_hrp(pdf, port, vp):
    hrp_info_pdf = get_hrp_prefixes(pdf, port, vp)
    hrp_pdf = hrp_info_pdf[hrp_info_pdf["is_hrp"] == True].copy()
    hrp_pdf.drop(columns=["is_hrp"], inplace=True)
    hrp_pdf["prefix"] = hrp_pdf["saddr"].apply(lambda x: ".".join(x.split(".")[:3]) + ".0/24")
    return hrp_pdf

In [ ]:
def store_zgrab_input(dataset, top100, vp):
    # zmap v4 files are like: zmap_465_20251127123220.csv
    zmap_files = glob(f"../results/zmap/dataset=tcp-anycast/vp={vp}/**/month=12/**/zmap_*.csv", recursive=True)
    if top100:
        with open("../input/zmap/nmap_100ports", "rt", encoding="utf-8") as f:
            result100 = f.read()
            localhost_ports = [int (i) for i in result100.splitlines()]
    #zmap_pdf = pd.DataFrame(columns=["saddr", "port"])
    for zmap_file in zmap_files:
        # extracting the port from the file name
        port = int(zmap_file.split("_")[-2])
        if port == 6699:
            # this is an UDP port
            continue
        if top100 and port not in localhost_ports:
            # only the top100 nmap ports...
            continue
        zmap_port_pdf = pd.read_csv(zmap_file)
        zmap_port_pdf["port"] = port
        # no hrp
        zmap_pdf = remove_hrp(zmap_port_pdf, port, vp)
        #zmap_pdf = pd.concat([zmap_pdf, zmap_no_hrp_pdf[["saddr", "port"]]], ignore_index=True)
        print("port:", port, "before", len(zmap_port_pdf), "after", len(zmap_pdf))

        zmap_pdf["domain"] = ""
        zmap_pdf["tag"] = zmap_pdf["port"].apply(lambda x: set_zgrab_tag(x))
        zmap_pdf[["saddr", "domain", "tag", "port"]].to_csv(f"../input/zgrab/top100ports_{vp}/zgrab_input_{port}_{dataset}_v4.csv", index=False, header=False)

store_zgrab_input("tcp-anycast", top100=True, vp="au-syd")

port: 135 before 612117 after 355307
port: 2049 before 574428 after 317805
port: 13 before 612848 after 355992
port: 22 before 631152 after 368507
port: 49157 before 612137 after 354914
port: 25 before 622847 after 355818
port: 5800 before 615491 after 359367
port: 79 before 595136 after 338361
port: 5060 before 3478154 after 0
port: 5631 before 605984 after 349241
port: 1433 before 618195 after 360450
port: 5051 before 624884 after 368820
port: 5432 before 617472 after 360467
port: 49156 before 613443 after 356171
port: 23 before 607400 after 360246
port: 3306 before 628108 after 354843
port: 990 before 607109 after 351095
port: 1720 before 610916 after 354508
port: 8008 before 612963 after 355183
port: 4899 before 629906 after 371907
port: 873 before 567567 after 359291
port: 7070 before 613442 after 355381
port: 8000 before 627669 after 360702
port: 8009 before 608257 after 350453
port: 2121 before 586568 after 330091
port: 8888 before 637989 after 363234
port: 3000 before 632542 af

In [34]:
!cat ../input/zgrab/top100ports_au-syd/*.csv > ../input/zgrab/zgrab_input_100ports_au-syd_tcp-anycast_v4.csv

In [37]:
def store_zgrab_hrp_input(dataset, top100, vp):
    # zmap v4 files are like: zmap_465_20251127123220.csv
    zmap_files = glob(f"../results/zmap/dataset=tcp-anycast/vp={vp}/**/month=12/**/zmap_*.csv", recursive=True)
    if top100:
        with open("../input/zmap/nmap_100ports", "rt", encoding="utf-8") as f:
            result100 = f.read()
            localhost_ports = [int (i) for i in result100.splitlines()]

    for zmap_file in zmap_files:
        # extracting the port from the file name
        port = int(zmap_file.split("_")[-2])
        if port == 6699 or port == 113:
            # this is an UDP port
            continue
        if top100 and port not in localhost_ports:
            # only the top100 nmap ports...
            continue
        zmap_port_pdf = pd.read_csv(zmap_file)
        zmap_port_pdf["port"] = port
        # hrp
        zmap_pdf = include_hrp(zmap_port_pdf, port, vp)
        # 100 IPs per prefix
        top100_zmap_pdf = zmap_pdf.groupby("prefix").head(100).copy()

        top100_zmap_pdf["domain"] = ""
        top100_zmap_pdf["tag"] = top100_zmap_pdf["port"].apply(lambda x: set_zgrab_tag(x))
        print("port:", port, "before", len(zmap_port_pdf), "after", len(top100_zmap_pdf))
        top100_zmap_pdf[["saddr", "domain", "tag", "port"]].to_csv(f"../input/zgrab/top100ports_{vp}/hrp/zgrab_input_{port}_{dataset}_v4.csv", index=False, header=False)


store_zgrab_hrp_input("tcp-anycast", top100=True, vp="au-syd")

port: 135 before 612117 after 101000
port: 2049 before 574428 after 100900
port: 13 before 612848 after 101000
port: 22 before 631152 after 103300
port: 49157 before 612137 after 101100
port: 25 before 622847 after 105000
port: 5800 before 615491 after 100700
port: 79 before 595136 after 101000
port: 5060 before 3478154 after 1398200
port: 5631 before 605984 after 100900
port: 1433 before 618195 after 101400
port: 5051 before 624884 after 100700
port: 5432 before 617472 after 101000
port: 49156 before 613443 after 101100
port: 23 before 607400 after 97200
port: 3306 before 628108 after 107400
port: 990 before 607109 after 100700
port: 1720 before 610916 after 100800
port: 8008 before 612963 after 101400
port: 4899 before 629906 after 101500
port: 873 before 567567 after 81900
port: 7070 before 613442 after 101500
port: 8000 before 627669 after 105000
port: 8009 before 608257 after 101400
port: 2121 before 586568 after 100900
port: 8888 before 637989 after 108000
port: 3000 before 63254

In [38]:
!cat ../input/zgrab/top100ports_au-syd/hrp/*.csv > ../input/zgrab/zgrab_input_hrp_100ports_au-syd_tcp-anycast_v4.csv

In [ ]:
# overview of zgrab input

files = [
    "../input/zgrab/zgrab_input_hrp_100ports_nl-ens_tcp-anycast_v4.csv",
    "../input/zgrab/zgrab_input_100ports_nl-ens_tcp-anycast_v4.csv",
    "../input/zgrab/zgrab_input_hrp_100ports_au-syd_tcp-anycast_v4.csv",
    "../input/zgrab/zgrab_input_100ports_au-syd_tcp-anycast_v4.csv"
]

scanners_set = set()
protocol_ports_set = set()
banner_ports_set = set()

for file in files:
    zgrab_input_pdf = pd.read_csv(file, names=["saddr", "domain", "tag", "port"])

    scanners_set.update(set(zgrab_input_pdf["tag"].to_list()))
    protocol_ports_set.update(set(zgrab_input_pdf[~zgrab_input_pdf["tag"].isin(["banner", "bannertls"])]["port"].to_list()))
    banner_ports_set.update(set(zgrab_input_pdf[zgrab_input_pdf["tag"].isin(["banner", "bannertls"])]["port"].to_list()))

print("protocols:", scanners_set)
print("protocol ports:", len(protocol_ports_set))
print("banner(tls) ports:", len(banner_ports_set))

protocols: {'banner', 'ipp', 'pop3', 'imaps', 'pop3s', 'postgres', 'ssh', 'pptp', 'smtps', 'http', 'ftps', 'imap', 'telnet', 'mysql', 'https', 'ftp'}
protocol ports: 23
banner(tls) ports: 77


In [19]:
files = [
    "../input/zgrab/zgrab_input_hrp_100ports_nl-ens_tcp-anycast_v4.csv",
    "../input/zgrab/zgrab_input_100ports_nl-ens_tcp-anycast_v4.csv",
    "../input/zgrab/zgrab_input_hrp_100ports_au-syd_tcp-anycast_v4.csv",
    "../input/zgrab/zgrab_input_100ports_au-syd_tcp-anycast_v4.csv"
]

for file in files:
    zgrab_input_pdf = pd.read_csv(file, names=["saddr", "domain", "tag", "port"])
    break

In [22]:
http_ports = set(zgrab_input_pdf[zgrab_input_pdf["tag"] == "http"]["port"].to_list())
print("HTTP ports:", http_ports)

HTTP ports: {8000, 8008, 5800, 80, 8080, 3128}


In [ ]:
zgrab_input_pdf

### NMap allowlist

reuses zgrab allowlist

In [6]:
vps = ["au-syd", "nl-ens"]

for vp in vps:
    zgrab_nohrp_pdf = pd.read_csv(f"../input/zgrab/zgrab_input_100ports_{vp}_tcp-anycast_v4.csv", header=None)
    zgrab_nohrp_pdf.columns = ["saddr", "domain", "tag", "port"]
    nmap_nohrp_pdf = zgrab_nohrp_pdf[zgrab_nohrp_pdf["port"] == 80]["saddr"].copy()

    zgrab_hrp_pdf = pd.read_csv(f"../input/zgrab/zgrab_input_hrp_100ports_{vp}_tcp-anycast_v4.csv", header=None)
    zgrab_hrp_pdf.columns = ["saddr", "domain", "tag", "port"]
    nmap_hrp_pdf = zgrab_hrp_pdf[zgrab_hrp_pdf["port"] == 80]["saddr"].copy()

    nmap_pdf = pd.concat([nmap_nohrp_pdf, nmap_hrp_pdf], ignore_index=True).drop_duplicates().reset_index(drop=True)

    nmap_pdf.to_csv(f"../input/nmap/nmap_input_100ports_{vp}_tcp-anycast_v4.csv", index=False, header=False)

In [18]:
# am I covering all anycast IPv4 prefixes?
vps = ["au-syd", "nl-ens"]
ts = datetime(2025, 12, 2)

_anycast_ipv4_pdf = pd.read_csv(f"../anycast_census_{ts.year}_{ts.month:02d}_{ts.day:02d}_v4.csv")
anycast_ipv4_pdf = _anycast_ipv4_pdf[_anycast_ipv4_pdf["GCD_ICMPv4"] > 1].copy()
prefixes = set(anycast_ipv4_pdf["prefix"].to_list())

for vp in vps:
    print(vp)
    nmap_pdf = pd.read_csv(f"../input/nmap/nmap_input_100ports_{vp}_tcp-anycast_v4.csv", header=None)
    nmap_pdf.columns = ["saddr"]
    ips = set(nmap_pdf["saddr"].to_list())

    missing_prefixes = []
    for prefix in prefixes:
        found = False
        prefix_n = ipaddress.IPv4Network(prefix.strip())
        for ip in ips:
            ip_n = ipaddress.IPv4Address(ip.strip())
            if ip_n in prefix_n:
                found = True
                break
        if not found:
            missing_prefixes.append(prefix)

    print("missing prefixes:", len(missing_prefixes), missing_prefixes)

au-syd
missing prefix 72.18.77.0/24
missing prefix 85.203.37.0/24
missing prefix 194.231.220.0/24
missing prefix 126.209.0.0/24
missing prefix 65.22.95.0/24
missing prefix 185.124.163.0/24
missing prefix 65.22.128.0/24
missing prefix 212.32.45.0/24
missing prefix 64.78.204.0/24
missing prefix 23.253.219.0/24
missing prefix 95.101.36.0/24
missing prefix 43.230.49.0/24
missing prefix 173.201.79.0/24
missing prefix 52.223.117.0/24
missing prefix 41.87.126.0/24
missing prefix 173.194.224.0/24
missing prefix 207.223.170.0/24
missing prefix 216.98.98.0/24
missing prefix 173.246.100.0/24
missing prefix 45.54.99.0/24
missing prefix 97.74.101.0/24
missing prefix 220.233.1.0/24
missing prefix 148.163.196.0/24
missing prefix 185.49.142.0/24
missing prefix 173.194.109.0/24
missing prefix 157.185.151.0/24
missing prefix 125.208.39.0/24
missing prefix 85.128.191.0/24
missing prefix 207.230.65.0/24
missing prefix 17.253.200.0/24
missing prefix 204.10.51.0/24
missing prefix 74.125.254.0/24
missing pre

KeyboardInterrupt: 

In [3]:
# I missed the other ports... let's see which IP addresses I missed to add to the nmap allow list file...

vps = ["au-syd", "nl-ens"]

for vp in vps:
    nmap80_pdf = pd.read_csv(f"../input/nmap/nmap_input_100ports_{vp}_tcp-anycast_v4.csv", header=None)
    nmap80_pdf.columns = ["saddr"]
    print(f"VP: {vp}, Nmap port 80 count: {len(nmap80_pdf)}")

    zgrab_nohrp_pdf = pd.read_csv(f"../input/zgrab/zgrab_input_100ports_{vp}_tcp-anycast_v4.csv", header=None)
    zgrab_nohrp_pdf.columns = ["saddr", "domain", "tag", "port"]
    nmap_nohrp_pdf = zgrab_nohrp_pdf.drop_duplicates(subset=["saddr"]).copy()
    zgrab_hrp_pdf = pd.read_csv(f"../input/zgrab/zgrab_input_hrp_100ports_{vp}_tcp-anycast_v4.csv", header=None)
    zgrab_hrp_pdf.columns = ["saddr", "domain", "tag", "port"]
    nmap_hrp_pdf = zgrab_hrp_pdf.drop_duplicates(subset=["saddr"]).copy()
    nmap_pdf = pd.concat([nmap_nohrp_pdf[["saddr"]], nmap_hrp_pdf[["saddr"]]], ignore_index=True).drop_duplicates(subset=["saddr"])
    print(f"VP: {vp}, Nmap all ports count: {len(nmap_pdf)}")

    nmap80_set = set(nmap80_pdf["saddr"].to_list())
    nmap_set = set(nmap_pdf["saddr"].to_list())
    missed_set = nmap_set - nmap80_set
    print(f"VP: {vp}, Missed count: {len(missed_set)}")
    pd.DataFrame({"saddr": list(missed_set)}).to_csv(f"../input/nmap/missed_nmap_ips_{vp}_tcp-anycast_v4.csv", index=False, header=False)

VP: au-syd, Nmap port 80 count: 1040548
VP: au-syd, Nmap all ports count: 1785628
VP: au-syd, Missed count: 745080
VP: nl-ens, Nmap port 80 count: 1071470
VP: nl-ens, Nmap all ports count: 1438411
VP: nl-ens, Missed count: 366941


In [7]:
# dividing into shards to run nmap in parallel if necessary
nshards = 30

for vp in vps:
    nmap_pdf = pd.read_csv(f"../input/nmap/nmap_input_100ports_{vp}_tcp-anycast_v4.csv", header=None)
    nmap_pdf.columns = ["saddr"]
    for shard in range(nshards):
        shard_pdf = nmap_pdf.iloc[shard::nshards]
        shard_pdf.to_csv(f"../input/nmap/sharding/{vp}/responsive_anycast_ipv4_addresses_shard_{shard+1}_of_{nshards}.csv", index=False, header=False)

In [8]:
# confirming that shards don't overlap
nshards = 30
for vp in vps:
    all_ips = set()
    for shard in range(nshards):
        shard_pdf = pd.read_csv(f"../input/nmap/sharding/{vp}/responsive_anycast_ipv4_addresses_shard_{shard+1}_of_{nshards}.csv", header=None)
        shard_pdf.columns = ["ipv4"]
        shard_ips = set(shard_pdf["ipv4"].tolist())
        intersection = all_ips.intersection(shard_ips)
        assert len(intersection) == 0, f"Overlap found in shard {shard+1}, {intersection}"
        all_ips.update(shard_ips)

### NMap allowlist -- old pipeline

In [ ]:
# do not use ISI hitlist...

ts = datetime(2025, 8, 27)

In [ ]:
_anycast_ipv4_pdf = pd.read_csv(f"anycast_census_{ts.year}_{ts.month:02d}_{ts.day:02d}_v4.csv")
anycast_ipv4_pdf = _anycast_ipv4_pdf[_anycast_ipv4_pdf["GCD_ICMPv4"] > 1].copy()
display(anycast_ipv4_pdf.head(5))
print(len(anycast_ipv4_pdf))

,prefix,AB_ICMPv4,AB_TCPv4,AB_DNSv4,GCD_ICMPv4,GCD_TCPv4,partial,backing_prefix,ASN,locations
0,1.0.0.0/24,29,29,29,75,30,False,1.0.0.0/24,13335,"[{'city': 'Honolulu', 'code_country': 'US', 'i..."
1,1.1.1.0/24,29,29,0,74,31,False,1.1.1.0/24,13335,"[{'city': 'Bangalore', 'code_country': 'IN', '..."
3,1.10.10.0/24,2,0,2,2,0,False,1.10.10.0/24,148000,"[{'city': 'Mumbai', 'code_country': 'IN', 'id'..."
4,1.12.0.0/24,5,0,5,5,0,False,1.12.0.0/20,132203_45090,"[{'city': 'Cologne', 'code_country': 'DE', 'id..."
5,1.12.12.0/24,3,1,0,3,1,False,1.12.0.0/20,132203_45090,"[{'city': 'Hong Kong', 'code_country': 'HK', '..."


13361


In [ ]:
ipv4_pdf = pd.read_csv(f"internet_address_verfploeter_hitlist_it113w-{ts.year}{ts.month:02d}{ts.day:02d}.csv", header=None)
ipv4_pdf.columns = ["ipv4"]

display(ipv4_pdf.head(5))
print(len(ipv4_pdf))

,ipv4
0,1.0.0.0
1,1.0.4.1
2,1.0.5.1
3,1.0.6.1
4,1.0.7.1


5788894


In [ ]:
ipv4_pdf["prefix"] = ipv4_pdf["ipv4"].apply(lambda x: ".".join(x.split(".")[:3]) + ".0/24")

display(ipv4_pdf.head(5))

,ipv4,prefix
0,1.0.0.0,1.0.0.0/24
1,1.0.4.1,1.0.4.0/24
2,1.0.5.1,1.0.5.0/24
3,1.0.6.1,1.0.6.0/24
4,1.0.7.1,1.0.7.0/24


In [ ]:
responsive_ipv4_pdf = anycast_ipv4_pdf.merge(ipv4_pdf, on="prefix", how="inner")

display(responsive_ipv4_pdf.head(5))
print(len(responsive_ipv4_pdf))

,prefix,AB_ICMPv4,AB_TCPv4,AB_DNSv4,GCD_ICMPv4,GCD_TCPv4,partial,backing_prefix,ASN,locations,ipv4
0,1.0.0.0/24,29,29,29,75,30,False,1.0.0.0/24,13335,"[{'city': 'Honolulu', 'code_country': 'US', 'i...",1.0.0.0
1,1.1.1.0/24,29,29,0,74,31,False,1.1.1.0/24,13335,"[{'city': 'Bangalore', 'code_country': 'IN', '...",1.1.1.0
2,1.10.10.0/24,2,0,2,2,0,False,1.10.10.0/24,148000,"[{'city': 'Mumbai', 'code_country': 'IN', 'id'...",1.10.10.10
3,1.12.0.0/24,5,0,5,5,0,False,1.12.0.0/20,132203_45090,"[{'city': 'Cologne', 'code_country': 'DE', 'id...",1.12.0.0
4,1.12.12.0/24,3,1,0,3,1,False,1.12.0.0/20,132203_45090,"[{'city': 'Hong Kong', 'code_country': 'HK', '...",1.12.12.12


13355


In [ ]:
responsive_ipv4 = responsive_ipv4_pdf.drop_duplicates(subset=["ipv4"])["ipv4"].copy()
responsive_ipv4.to_csv(f"../input/nmap/responsive_anycast_ipv4_addresses_{ts.year}{ts.month:02d}{ts.day:02d}.csv", index=False, header=False)

In [ ]:
# dividing into shards to run nmap in parallel if necessary
nshards = 30

for shard in range(nshards):
    shard_pdf = responsive_ipv4.iloc[shard::nshards]
    shard_pdf.to_csv(f"../input/nmap/sharding/responsive_anycast_ipv4_addresses_shard_{shard+1}_of_{nshards}_{ts.year}{ts.month:02d}{ts.day:02d}.csv", index=False, header=False)

In [ ]:
# confirming that shards don't overlap
nshards = 30
all_ips = set()
for shard in range(nshards):
    shard_pdf = pd.read_csv(f"../input/nmap/sharding/responsive_anycast_ipv4_addresses_shard_{shard+1}_of_{nshards}_{ts.year}{ts.month:02d}{ts.day:02d}.csv", header=None)
    shard_pdf.columns = ["ipv4"]
    shard_ips = set(shard_pdf["ipv4"].tolist())
    intersection = all_ips.intersection(shard_ips)
    assert len(intersection) == 0, f"Overlap found in shard {shard+1}, {intersection}"
    all_ips.update(shard_ips)

In [8]:
def create_allowlist_from_zmap(dataset, ts, output_filename, input_to):
    # zmap v4 files are like: zmap_465_20251127123220_v4.csv
    zmap_v4_files = glob(f"../results/zmap/dataset={dataset}/**/zmap_*.csv", recursive=True)

    zmap_v4_pdf = pd.DataFrame(columns=["saddr", "port"])
    for zmap_v4_file in zmap_v4_files:
        # extracting the port from the file name
        port = int(zmap_v4_file.split("_")[-2])
        zmap_port_v4_pdf = pd.read_csv(zmap_v4_file)
        zmap_port_v4_pdf["port"] = port
        zmap_v4_pdf = pd.concat([zmap_v4_pdf, zmap_port_v4_pdf[["saddr", "port"]]], ignore_index=True)

    zmap_v4_pdf.drop_duplicates(["saddr"])["saddr"].to_csv(f"../input/{input_to}/{output_filename}_{ts.year}{ts.month:02d}{ts.day:02d}.csv", index=False, header=False)

In [ ]:
ts = datetime(2025, 11, 28)  # date when ZMap scan was done
create_allowlist_from_zmap("tcp-anycast", ts, "zmap_responsive_tcp_ipv4_addresses", "nmap")

In [9]:
# IPv4
# date when ZMap UDP scan was done
#vp="nl-ens"
#ts = datetime(2025, 12, 9)  # vp=nl-ens
vp="au-syd"
ts = datetime(2025, 12, 13)  # vp=au-syd
ports = [443, 853]
#create_allowlist_from_zmap("udp-anycast", ts, "zmap_responsive_udp_ipv4_addresses", "quic")

# zmap v4 files are like: zmap_465_20251127123220_v4.csv
for port in ports:
    zmap_v4_files = glob(f"../results/zmap/dataset=udp-anycast/vp={vp}/port={port}/year={ts.year}/month={ts.month:02d}/day={ts.day:02d}/zmap_*.csv")[0]
    # ip,hostname,port
    zmap_pdf = pd.read_csv(zmap_v4_files)
    zmap_pdf["hostname"] = ""
    zmap_pdf["port"] = port
    zmap_pdf.columns = ["ip", "hostname", "port"]
    zmap_pdf.to_csv(f"../input/quic/zmap_responsive_udp_{vp}_{port}_{ts.year}{ts.month:02d}{ts.day:02d}_v4.csv", index=False)


sysctl -n net.core.rmem_max  
212992

sysctl -n net.core.wmem_max  
212992

## IPv6

We use the file v6_20250818.csv as IPv6 hitlist which includes TUM v6 hitlist + OpenINTEL AAAA records (from czds, com, net and org).

We also use the oi_v6_20251215.csv from OpenINTEL AAAA records (all zones).  

### ZMap allowlist -- unused

Simply use the v6_20250818.csv hitlist

In [3]:
ts_v6 = datetime(2025, 12, 9)

In [ ]:
_anycast_ipv6_pdf = pd.read_csv(f"anycast_census_{ts_v6.year}_{ts_v6.month:02d}_{ts_v6.day:02d}_v6.csv")
anycast_ipv6_pdf = _anycast_ipv6_pdf[_anycast_ipv6_pdf["GCD_ICMPv6"] > 1].copy()
display(anycast_ipv6_pdf.head(5))
print(len(anycast_ipv6_pdf))
anycast_ipv6_pdf["prefix"].to_csv(f"input/anycast_prefixes_{ts_v6.year}_{ts_v6.month:02d}_{ts_v6.day:02d}_v6.csv", index=False, header=False)

,prefix,AB_ICMPv6,AB_TCPv6,AB_DNSv6,GCD_ICMPv6,GCD_TCPv6,backing_prefix,ASN,locations
0,2001:1201::/48,2,2,2,2,1,2001:1201::/48,28498,"[{'city': 'Querétaro', 'country_code': 'MX', '..."
11,2001:1258::/48,2,2,2,2,2,2001:1258::/32,28499,"[{'city': 'Querétaro', 'country_code': 'MX', '..."
13,2001:12f8:2::/48,3,0,3,3,0,2001:12f8:2::/48,12136,"[{'city': 'Singapore', 'country_code': 'SG', '..."
14,2001:12f8:4::/48,4,0,4,5,0,2001:12f8:4::/48,11644,"[{'city': None, 'country_code': None, 'id': 'N..."
15,2001:12f8:8::/48,3,0,3,3,0,2001:12f8:8::/48,10906,"[{'city': 'St Petersburg', 'country_code': 'RU..."


12821


### NMap allowlist

In [53]:
ts_v6 = datetime(2025, 12, 15)

In [54]:
_anycast_ipv6_pdf = pd.read_csv(f"../anycast_census_{ts_v6.year}_{ts_v6.month:02d}_{ts_v6.day:02d}_v6.csv")
anycast_ipv6_pdf = _anycast_ipv6_pdf[_anycast_ipv6_pdf["GCD_ICMPv6"] > 1].copy()
display(anycast_ipv6_pdf.head(5))
print(len(anycast_ipv6_pdf))

,prefix,AB_ICMPv6,AB_TCPv6,AB_DNSv6,GCD_ICMPv6,GCD_TCPv6,backing_prefix,ASN,locations
0,2001:1201:10::/48,3,3,2,2,1,2001:1201:10::/48,27661,"[{'city': 'Vancouver', 'country_code': 'CA', '..."
1,2001:1201::/48,4,4,4,2,2,2001:1201::/48,28498,"[{'city': 'Querétaro', 'country_code': 'MX', '..."
111,2001:1258::/48,3,3,3,2,1,2001:1258::/32,28499,"[{'city': 'Querétaro', 'country_code': 'MX', '..."
113,2001:12f8:2::/48,3,0,3,3,0,2001:12f8:2::/48,12136,"[{'city': 'Singapore', 'country_code': 'SG', '..."
114,2001:12f8:4::/48,4,0,4,5,0,2001:12f8:4::/48,11644,"[{'city': None, 'country_code': None, 'id': 'N..."


13066


In [49]:
r_ipv6_pdf = pd.read_csv(f"../v6_20250818.csv", header=None)

r_ipv6_pdf.columns = ["ipv6"]
display(r_ipv6_pdf.head(5))
print(len(r_ipv6_pdf))

,ipv6
0,2620:10a:80ac::45
1,2001:678:2c:0:194:0:28:7
2,2001:678:78:42:ad::53
3,2001:dce:2000:2::130
4,2001:dce:7000:2::130


6234585


In [51]:
oi_ipv6_pdf = pd.read_csv("../oi_v6_20251215.csv", header=None)
oi_ipv6_pdf.columns = ["ipv6"]
display(oi_ipv6_pdf.head(5))
print(len(oi_ipv6_pdf))

,ipv6
0,2001:41d0:301::30
1,2a02:4780:23:399e:5d41:7a82:9063:6f6c
2,2606:4700:3031::6815:d27
3,2a06:41c0:1:10::1c6
4,2a02:4780:23:40d1:87d1:aad5:16b1:7592


17787237


In [50]:
ipv6_pdf = pd.concat([r_ipv6_pdf, oi_ipv6_pdf], ignore_index=True).drop_duplicates(subset=["ipv6"]).copy()
display(ipv6_pdf.head(5))
print(len(ipv6_pdf))

,ipv6
0,2620:10a:80ac::45
1,2001:678:2c:0:194:0:28:7
2,2001:678:78:42:ad::53
3,2001:dce:2000:2::130
4,2001:dce:7000:2::130


23931717


In [55]:
anycast_prefixes = anycast_ipv6_pdf["prefix"].tolist()
pyt = pytricia.PyTricia(128)
for prefix in anycast_prefixes:
    pyt.insert(ipaddress.IPv6Network(prefix), True)

ipv6_pdf["is_anycasted"] = ipv6_pdf["ipv6"].apply(lambda x: ipaddress.IPv6Address(x) in pyt)

In [56]:
ipv6_pdf["is_anycasted"].value_counts()

is_anycasted
False    23771967
True       159750
Name: count, dtype: int64

In [57]:
ipv6_pdf[ipv6_pdf["is_anycasted"] == True][["ipv6"]].to_csv(f"../input/nmap/responsive_anycast_ipv6_addresses_{ts_v6.year}{ts_v6.month:02d}{ts_v6.day:02d}.csv", index=False, header=False)

sharding

In [60]:
# dividing into shards to run nmap in parallel if necessary
nshards = 30

for shard in range(nshards):
    shard_pdf = ipv6_pdf[ipv6_pdf["is_anycasted"] == True]["ipv6"].iloc[shard::nshards]
    shard_pdf.to_csv(f"../input/nmap/shardingv6/responsive_anycast_ipv6_addresses_shard_{shard+1}_of_{nshards}_{ts_v6.year}{ts_v6.month:02d}{ts_v6.day:02d}.csv", index=False, header=False)

In [61]:
# confirming that shards don't overlap
nshards = 30
all_ips = set()
for shard in range(nshards):
    shard_pdf = pd.read_csv(f"../input/nmap/shardingv6/responsive_anycast_ipv6_addresses_shard_{shard+1}_of_{nshards}_{ts_v6.year}{ts_v6.month:02d}{ts_v6.day:02d}.csv", header=None)
    shard_pdf.columns = ["ipv6"]
    shard_ips = set(shard_pdf["ipv6"].tolist())
    intersection = all_ips.intersection(shard_ips)
    assert len(intersection) == 0, f"Overlap found in shard {shard+1}, {intersection}"
    all_ips.update(shard_ips)

### ZGrab

In [58]:
ts_v6 = datetime(2025, 12, 15)

with open("../input/zmap/nmap_100ports", "rt", encoding="utf-8") as f:
    result100 = f.read()
    localhost_ports = [int (i) for i in result100.splitlines()]

nmap_pdf = pd.DataFrame(columns=["ipv6", "port", "domain", "tag"])
for port in localhost_ports:
    _nmap_pdf = pd.read_csv(f"../input/nmap/responsive_anycast_ipv6_addresses_{ts_v6.year}{ts_v6.month:02d}{ts_v6.day:02d}.csv", header=None)
    _nmap_pdf.columns = ["ipv6"]
    _nmap_pdf["port"] = port
    _nmap_pdf["domain"] = ""
    _nmap_pdf["tag"] = _nmap_pdf["port"].apply(lambda x: set_zgrab_tag(x))
    nmap_pdf = pd.concat([nmap_pdf, _nmap_pdf], ignore_index=True)

dataset = "tcp-anycast"
nmap_pdf[["ipv6", "domain", "tag", "port"]].to_csv(f"../input/zgrab/zgrab_input_{dataset}_v6.csv", index=False, header=False)


### QUIC allowlist

In [59]:
ts_v6 = datetime(2025, 12, 15)
ports = [443, 853]
ipv6_anycasted_pdf = pd.read_csv(f"../input/nmap/responsive_anycast_ipv6_addresses_{ts_v6.year}{ts_v6.month:02d}{ts_v6.day:02d}.csv", header=None)
for port in ports:
    ipv6_anycasted_pdf["hostname"] = ""
    ipv6_anycasted_pdf["port"] = port
    ipv6_anycasted_pdf.columns = ["ip", "hostname", "port"]
    ipv6_anycasted_pdf.to_csv(f"../input/quic/responsive_{port}_{ts_v6.year}{ts_v6.month:02d}{ts_v6.day:02d}_v6.csv", index=False)

# Process output

## Process ZMap output

In [31]:
def process_zmap_anycast_results(dataset):
    # zmap v4 files are like: zmap_465_20251127123220.csv
    zmap_v4_files = glob(f"../results/zmap/dataset={dataset}/**/zmap_*.csv", recursive=True)

    zmap_v4_pdf = pd.DataFrame(columns=["saddr", "port"])
    for zmap_v4_file in zmap_v4_files:
        print(f"Processing {zmap_v4_file}...")
        # extracting the port from the file name
        port = int(zmap_v4_file.split("_")[-2])
        zmap_port_v4_pdf = pd.read_csv(zmap_v4_file)
        zmap_port_v4_pdf["port"] = port
        zmap_v4_pdf = pd.concat([zmap_v4_pdf, zmap_port_v4_pdf[["saddr", "port"]]], ignore_index=True)

    display(
        zmap_v4_pdf.groupby("port").size().reset_index(name="counts").sort_values(by="counts", ascending=False)
    )

    print("unique reponsive IPv4 addresses:", zmap_v4_pdf["saddr"].nunique())

In [ ]:
# far too many results to download and process here.. going to cluster..
process_zmap_anycast_results("tcp-anycast")

In [32]:
process_zmap_anycast_results("udp-anycast")

Processing ../results/zmap/dataset=udp-anycast/vp=nl-ens/port=443/year=2025/month=12/day=09/zmap_443_20251209140023.csv...
Processing ../results/zmap/dataset=udp-anycast/vp=nl-ens/port=853/year=2025/month=12/day=09/zmap_853_20251209142140.csv...


,port,counts
0,443,1055004
1,853,1326


unique reponsive IPv4 addresses: 1055029


## Process nmap output

In [2]:
#xml_files = glob("../results/nmap/production_sharded_scansv6/*.xml")
xml_files = ["../results/nmap/nmap_20251216201453__22_of_30.csv_.xml"]  # for testing

rows = []  # collect rows here
scan_time = 0
for xml_file in xml_files:
    #print(f"Processing {xml_file}...")

    # parse rtt for each host
    try:
        tree = ET.parse(xml_file)
    except ET.ParseError as e:
        print(f"Error parsing {xml_file}: {e}")
        continue
    root = tree.getroot()

    host_rtts = {}
    cnt = 0
    if root.find("runstats") is None:
        scan_time += 0
    else:
        scan_time += float(root.find("runstats").find("finished").get("elapsed"))
    for host in root.findall("host"):
        addr_el = host.find("address")
        ip = addr_el.get("addr") if addr_el is not None else None

        times = host.find("times")
        if times is not None:
            srtt = int(times.get("srtt")) / 1e6   # convert to s
            host_rtts[ip] = srtt
            cnt += 1

    #print("amount of srtt found:", cnt)
    report = NmapParser.parse_fromfile(xml_file)
    filename = os.path.splitext(os.path.basename(xml_file))[0]

    scan_ts = pd.to_datetime(filename.split("_")[1], utc=True)
    for host in report.hosts:
        for svc in host.services:
            rows.append(
            {
                "host": host.address,
                "rtt_ms": host_rtts.get(host.address),
                "port": svc.port,
                "protocol": svc.protocol,
                "service": svc.service,
                "os": host.os,
                "banner": svc.banner,
                "scan_timestamp": scan_ts,
            })
            #nmap_pdf = pd.concat([nmap_pdf, row], ignore_index=True)
    uniq_hosts_w_services = set([host.address for host in report.hosts if len(host.services) > 0])
    uniq_hosts = set([host.address for host in report.hosts])
    print("file", xml_file, "uniq hosts", len(uniq_hosts), "hosts with services", len(uniq_hosts_w_services))

    if False:
        with open(f"../results/nmap/production_sharded_scans/{filename}.csv", "wt", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["host", "rtt_ms", "port", "protocol", "service", "os", "banner"])
            for host in report.hosts:
                #print(dir(host))
                for svc in host.services:
                    writer.writerow([
                        host.address,
                        host_rtts.get(host.address),
                        svc.port,
                        svc.protocol,
                        svc.service,
                        host.os,
                        svc.banner,
                    ])

nmap_expanded_pdf = pd.DataFrame(rows, columns=["host", "rtt_ms", "port", "protocol",
                                       "service", "os", "banner", "scan_timestamp"])
#nmap_expanded_pdf.to_csv("../results/nmap/production_sharded_scans/nmap_combined_production_sharded_scans.csv", index=False)
print("Processed files:", len(xml_files))
print("Total scan time if not sharded:", scan_time/3600, "hours")
print("Average scan time per file:", scan_time / len(xml_files) / 3600, "hours")
#print("hosts with service", nmap_expanded_pdf["host"].nunique())

file ../results/nmap/nmap_20251216201453__22_of_30.csv_.xml uniq hosts 35715 hosts with services 20708
Processed files: 1
Total scan time if not sharded: 322.5801055555555 hours
Average scan time per file: 322.5801055555555 hours
hosts with service 20708


In [31]:
# tab delimited file
nmap_services_pdf = pd.read_csv("../nmap-services", delimiter="\t")
nmap_services_pdf.columns = ["service_name", "portnum_protocol", "open_frequency", "optional_comments"]
nmap_services_pdf.sort_values(by=["open_frequency"], ascending=False, inplace=True)
default_nmap_services_pdf = nmap_services_pdf.head(1000).copy()

In [21]:
# https://secwiki.org/w/FAQ_tcpwrapped

In [ ]:
#nmap_expanded_pdf = pd.read_csv("../results/nmap/production_sharded_scans/nmap_combined_production_sharded_scans.csv")

In [5]:
cols_to_list = ["protocol", "service", "os", "banner", "port"]
nmap_pdf = nmap_expanded_pdf.groupby(["host", "rtt_ms", "scan_timestamp"], dropna=False).agg({col: list for col in cols_to_list}).reset_index()

display(nmap_pdf)
#display(nmap_pdf.groupby([ "service"]).size().reset_index(name="counts").sort_values(by="counts", ascending=False))

print("hosts with service", nmap_pdf["host"].nunique())

nmap_pdf["unique_services"] = nmap_pdf["service"].apply(lambda x: set(x))
display(nmap_pdf.explode("unique_services").groupby("unique_services").size().reset_index(name="counts").sort_values(by="counts", ascending=False))

ports = set(nmap_pdf.explode("port")["port"].to_list())
print("unique ports scanned:", len(ports))

services = set(nmap_pdf.explode("unique_services")["unique_services"].to_list())
print("unique services scanned:", len(services), services)

,host,rtt_ms,scan_timestamp,protocol,service,os,banner,port
0,1.0.0.159,0.004226,2025-12-16 20:14:53+00:00,"[tcp, tcp, tcp, tcp]","[http, https, http, https-alt]","[Fingerprints: , Fingerprints: , Fingerprints:...","[product: Cloudflare http proxy, product: clou...","[80, 443, 8080, 8443]"
1,1.0.0.249,0.004420,2025-12-16 20:14:53+00:00,"[tcp, tcp, tcp, tcp]","[http, https, http, https-alt]","[Fingerprints: , Fingerprints: , Fingerprints:...","[product: Cloudflare http proxy, product: clou...","[80, 443, 8080, 8443]"
2,1.0.0.57,0.004339,2025-12-16 20:14:53+00:00,"[tcp, tcp, tcp, tcp]","[http, https, http, https-alt]","[Fingerprints: , Fingerprints: , Fingerprints:...","[product: Cloudflare http proxy, product: clou...","[80, 443, 8080, 8443]"
3,1.1.1.141,0.004147,2025-12-16 20:14:53+00:00,"[tcp, tcp, tcp, tcp]","[http, https, http, https-alt]","[Fingerprints: , Fingerprints: , Fingerprints:...","[product: Cloudflare http proxy, product: clou...","[80, 443, 8080, 8443]"
4,1.1.1.30,0.004311,2025-12-16 20:14:53+00:00,"[tcp, tcp, tcp, tcp]","[http, https, http, https-alt]","[Fingerprints: , Fingerprints: , Fingerprints:...","[product: Cloudflare http proxy, product: clou...","[80, 443, 8080, 8443]"
...,...,...,...,...,...,...,...,...
20703,99.83.204.245,0.003931,2025-12-16 20:14:53+00:00,"[tcp, tcp]","[http, https]","[Fingerprints: , Fingerprints: ]","[product: awselb/2.0, product: awselb/2.0]","[80, 443]"
20704,99.83.207.185,0.003883,2025-12-16 20:14:53+00:00,"[tcp, tcp, tcp, tcp]","[http, ldap, http, ldap]","[Fingerprints: , Fingerprints: , Fingerprints:...","[product: awselb/2.0, , product: nginx, ]","[80, 389, 443, 636]"
20705,99.83.212.242,0.003829,2025-12-16 20:14:53+00:00,"[tcp, tcp]","[http, https]","[Fingerprints: , Fingerprints: ]","[product: awselb/2.0, product: awselb/2.0]","[80, 443]"
20706,99.83.213.164,0.003776,2025-12-16 20:14:53+00:00,"[tcp, tcp]","[tcpwrapped, http]","[Fingerprints: , Fingerprints: ]","[, product: Microsoft IIS httpd version: 10.0 ...","[80, 443]"


hosts with service 20708


,unique_services,counts
204,https,17182
199,http,16993
205,https-alt,8811
467,tcpwrapped,4960
202,http-proxy,878
...,...,...
306,nameserver,1
305,nagios-nsca,1
304,n1-fwp,1
303,mythtv,1


unique ports scanned: 1000
unique services scanned: 545 {'svrloc', 'netvenuechat', 'flexlm0', 'ldap', 'pxc-spvr', 'cadlock', 'snet-sensor-mgmt', 'raid-am', 'kyoceranetdev', 'ppp', 'pktcablemmcops', 'sometimes-rpc7', 'interbase', 'pcduo', 'kshell', 'rsync', 'wormux', 'tmi', 'instl_bootc', 'iphone-sync', 'ospf-lite', 'connection', 'lotusnotes', 'hp-hcip', 'gris', 'time', 'nicetec-mgmt', 'microsoft-ds', 'pxc-splr-ft', 'sgi-soap', 'rdrmshc', 'iris-xpcs', 'fodms', 'enl-name', 'isakmp', 'msdtc', 'dnp', 'dc', 'sane-port', 'accessbuilder', 'veracity', 'ida-agent', 'ssh', 'pn-requester', 'apc-agent', 'X11:3', 'ansoft-lm-1', 'timbuktu-srv1', 'glrpc', 'mit-ml-dev', 'mdbs_daemon', 'ftp-data', 'td-postman', 'funk-dialout', 'excw', 'ospfd', 'uucp-rlogin', 'sms-rcinfo', 'ldapssl', 'nati-svrloc', 'upnp', 'documentum_s', 'cnrprotocol', 'x25-svc-port', 'https', 'ies-lm', 'cce4x', 'cisco-sccp', 'tn-tl-r1', 'mysql-cm-agent', 'hpvmmcontrol', 'nut', 'nsjtp-data', 'xprint-server', 'pn-requester2', 'trim', 'w

In [ ]:
# TEST SCAN!
nmap_pdf = pd.read_csv("results/nmap/output.csv")
display(nmap_pdf)
print(len(nmap_pdf))

,host,rtt_ms,port,protocol,service,os,banner
0,1.0.0.0,1.212,80,tcp,http,Fingerprints:,product: Cloudflare http proxy
1,1.0.0.0,1.212,443,tcp,https,Fingerprints:,product: cloudflare
2,1.0.0.0,1.212,8080,tcp,http,Fingerprints:,product: Cloudflare http proxy
3,1.0.0.0,1.212,8443,tcp,https-alt,Fingerprints:,product: cloudflare
4,1.1.1.0,1.250,80,tcp,http,Fingerprints:,product: Cloudflare http proxy
5,1.1.1.0,1.250,443,tcp,https,Fingerprints:,product: cloudflare
6,1.1.1.0,1.250,8080,tcp,http,Fingerprints:,product: Cloudflare http proxy
7,1.1.1.0,1.250,8443,tcp,https-alt,Fingerprints:,product: cloudflare
8,1.1.8.8,207.842,80,tcp,http,Fingerprints:,product: nginx version: 1.20.1
9,1.1.8.8,207.842,443,tcp,http,Fingerprints:,product: OpenResty web app server


24


## Process mtr output

In [34]:
mtr_files = glob("results/mtr/*.csv")
for mtr_file in mtr_files:
    print(f"Processing {mtr_file}...")
    # Add your processing code here
    mtr_pdf = pd.read_csv(mtr_file)
    display(mtr_pdf)

Processing results/mtr/mtr_1.12.15.31.csv...


,hostname,Mtr_Version,Start_Time,Status,Host,Hop,Ip,Asn,Loss%,Snt,,Last,Avg,Best,Wrst,StDev
0,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,1,172.18.0.1 (172.18.0.1),AS???,0.0,1,0,0.33,0.33,0.33,0.33,0.0
1,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,2,169.254.169.254 (169.254.169.254),AS???,0.0,1,0,0.20,0.20,0.20,0.20,0.0
2,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,3,vl199-ds1-j2-51917.ams1.constant.com (45.76.40...,AS20473,0.0,1,0,18.60,18.60,18.60,18.60,0.0
3,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,4,10.72.0.5 (10.72.0.5),AS???,0.0,1,0,1.78,1.78,1.78,1.78,0.0
4,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,5,10.72.4.13 (10.72.4.13),AS???,0.0,1,0,0.95,0.95,0.95,0.95,0.0
5,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,6,5-1-40.ear3.amsterdam1.level3.net (213.19.196....,AS3356,0.0,1,0,1.63,1.63,1.63,1.63,0.0
6,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,7,???,AS???,100.0,1,1,0.00,0.00,0.00,0.00,0.0
7,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,8,62.67.67.70 (62.67.67.70),AS3356,0.0,1,0,8.88,8.88,8.88,8.88,0.0
8,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,9,???,AS???,100.0,1,1,0.00,0.00,0.00,0.00,0.0
9,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,10,???,AS???,100.0,1,1,0.00,0.00,0.00,0.00,0.0
